In [2]:
import pandas as pd
import numpy as np


df_clean = pd.read_excel("../data/telco_churn_clean.xlsx")

In [3]:
# 1. Programmatically identify columns that have exactly 2 unique values
binary_columns = [
    col for col in df_clean.columns 
    if df_clean[col].dropna().nunique() == 2
]

print("Columns identified as strictly binary:", binary_columns)

# 2. Define explicit mapping dictionaries for our variants
binary_mappings = {
    # Yes/No Mapping
    'Yes': 1, 'No': 0,
    # Gender Mapping
    'Male': 1, 'Female': 0
}

# 3. Apply the mapping across all identified binary columns
for col in binary_columns:
    # .map() checks if values match our keys; handles safe replacement
    df_clean[col] = df_clean[col].map(binary_mappings).fillna(df_clean[col])

# Verify the changes
print(df_clean[binary_columns].head())

df_clean.head()


Columns identified as strictly binary: ['Gender', 'Senior Citizen', 'Partner', 'Dependents', 'Phone Service', 'Paperless Billing', 'Churn Value']
   Gender  Senior Citizen  Partner  Dependents  Phone Service  \
0       1               0        0           0              1   
1       0               0        0           1              1   
2       0               0        0           1              1   
3       0               0        1           1              1   
4       1               0        0           1              1   

   Paperless Billing  Churn Value  
0                  1          1.0  
1                  1          1.0  
2                  1          1.0  
3                  1          1.0  
4                  1          1.0  


,City,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Multiple Lines,Internet Service,Online Security,...,Tech Support,Streaming TV,Streaming Movies,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Value,CLTV
0,Los Angeles,1,0,0,0,2,1,No,DSL,Yes,...,No,No,No,Month-to-month,1,Mailed check,53.85,108.15,1.0,3239
1,Los Angeles,0,0,0,1,2,1,No,Fiber optic,No,...,No,No,No,Month-to-month,1,Electronic check,70.70,151.65,1.0,2701
2,Los Angeles,0,0,0,1,8,1,Yes,Fiber optic,No,...,No,Yes,Yes,Month-to-month,1,Electronic check,99.65,820.50,1.0,5372
3,Los Angeles,0,0,1,1,28,1,Yes,Fiber optic,No,...,Yes,Yes,Yes,Month-to-month,1,Electronic check,104.80,3046.05,1.0,5003
4,Los Angeles,1,0,0,1,49,1,Yes,Fiber optic,No,...,No,Yes,Yes,Month-to-month,1,Bank transfer (automatic),103.70,5036.30,1.0,5340


In [4]:
multi_cat_cols = [
    'Multiple Lines', 'Internet Service', 'Online Security', 
    'Online Backup', 'Device Protection', 'Tech Support', 
    'Streaming TV', 'Streaming Movies', 'Contract', 'Payment Method'
]

df_encoded = pd.get_dummies(df_clean, columns=multi_cat_cols, drop_first=True, dtype=int)
df_encoded.head()

,City,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Paperless Billing,Monthly Charges,Total Charges,...,Tech Support_Yes,Streaming TV_No internet service,Streaming TV_Yes,Streaming Movies_No internet service,Streaming Movies_Yes,Contract_One year,Contract_Two year,Payment Method_Credit card (automatic),Payment Method_Electronic check,Payment Method_Mailed check
0,Los Angeles,1,0,0,0,2,1,1,53.85,108.15,...,0,0,0,0,0,0,0,0,0,1
1,Los Angeles,0,0,0,1,2,1,1,70.70,151.65,...,0,0,0,0,0,0,0,0,1,0
2,Los Angeles,0,0,0,1,8,1,1,99.65,820.50,...,0,0,1,0,1,0,0,0,1,0
3,Los Angeles,0,0,1,1,28,1,1,104.80,3046.05,...,1,0,1,0,1,0,0,0,1,0
4,Los Angeles,1,0,0,1,49,1,1,103.70,5036.30,...,0,0,1,0,1,0,0,0,0,0


In [5]:
top_cities = df_encoded['City'].value_counts().nlargest(10).index
df_encoded['City'] = df_encoded['City'].where(df_encoded['City'].isin(top_cities), 'Other')
df_encoded = pd.get_dummies(df_encoded, columns=['City'], drop_first=True)

print(df_encoded.shape)
print(df_encoded.dtypes.value_counts())

(7043, 42)
int64      29
bool       10
float64     3
Name: count, dtype: int64


In [6]:
df_encoded.head()

,Gender,Senior Citizen,Partner,Dependents,Tenure Months,Phone Service,Paperless Billing,Monthly Charges,Total Charges,Churn Value,...,City_Glendale,City_Long Beach,City_Los Angeles,City_Oakland,City_Other,City_Sacramento,City_San Diego,City_San Francisco,City_San Jose,City_Stockton
0,1,0,0,0,2,1,1,53.85,108.15,1.0,...,False,False,True,False,False,False,False,False,False,False
1,0,0,0,1,2,1,1,70.70,151.65,1.0,...,False,False,True,False,False,False,False,False,False,False
2,0,0,0,1,8,1,1,99.65,820.50,1.0,...,False,False,True,False,False,False,False,False,False,False
3,0,0,1,1,28,1,1,104.80,3046.05,1.0,...,False,False,True,False,False,False,False,False,False,False
4,1,0,0,1,49,1,1,103.70,5036.30,1.0,...,False,False,True,False,False,False,False,False,False,False


In [7]:
from sklearn.model_selection import train_test_split

X = df_encoded.drop('Churn Value', axis=1)
y = df_encoded['Churn Value']

# First split: carve off 15% for test
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.15, random_state=42, stratify=y)

# Second split: from the remaining 85%, take 0.15/0.85 ≈ 0.176 for val
X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.15/0.85, random_state=42, stratify=y_train_val)


print(X_train.shape, X_val.shape, X_test.shape)


(4929, 41) (1057, 41) (1057, 41)


In [8]:
from sklearn.preprocessing import StandardScaler

numerical_cols = ['Tenure Months', 'Monthly Charges', 'Total Charges' , "CLTV"]
scaler = StandardScaler()

X_train[numerical_cols] = scaler.fit_transform(X_train[numerical_cols])
X_val[numerical_cols] = scaler.transform(X_val[numerical_cols])
X_test[numerical_cols] = scaler.transform(X_test[numerical_cols])

X_train[numerical_cols].describe()

,Tenure Months,Monthly Charges,Total Charges,CLTV
count,4.929000e+03,4.929000e+03,4.929000e+03,4.929000e+03
mean,-7.928556e-18,-2.027188e-16,1.890240e-16,2.425417e-16
std,1.000101e+00,1.000101e+00,1.000101e+00,1.000101e+00
min,-1.313275e+00,-1.530791e+00,-1.004614e+00,-2.033636e+00
25%,-9.490814e-01,-9.822720e-01,-8.336048e-01,-7.915310e-01
50%,-1.397630e-01,1.874613e-01,-3.908723e-01,1.102013e-01
75%,9.528168e-01,8.367624e-01,6.792751e-01,8.171934e-01
max,1.600271e+00,1.785106e+00,2.807455e+00,1.769728e+00


In [9]:
df_encoded.shape

(7043, 42)